# 00 · Broadcast UDP y grafo de descubrimiento

**Objetivo**: ver en vivo cómo los peers se descubren entre sí por broadcast UDP
y cómo el grafo de conectividad evoluciona con el tiempo.

Este notebook es **previo** al notebook 01. No asume conocimiento del código fuente;
solo simula el comportamiento observable desde afuera.

Necesitas: `pip install matplotlib networkx`

## 1. Elige tu peerId

En la red real, el peerId se genera al arrancar (`generatePeerId()` en `src/protocol.ts`).
Aquí puedes escribir el tuyo o dejar el campo vacío para que se genere uno aleatorio.

In [ ]:
import random
import string

def random_peer_id():
    """Imita el hex de SHA-256 de 32 bytes aleatorios (truncado a 16 chars para visualización)."""
    return ''.join(random.choices('0123456789abcdef', k=16))

raw = input("Tu peerId (Enter = aleatorio): ").strip()
MY_PEER_ID = raw if raw else random_peer_id()
print(f"peerId: {MY_PEER_ID}")

## 2. Define los demás peers de la red

Edita la lista `OTHER_PEERS` para simular cuántos vecinos tiene tu LAN.
Puedes poner nombres legibles o IDs hex — lo que quieras.

Parámetros que controlan el comportamiento:

In [ ]:
# ── Peers en la red (además del tuyo) ────────────────────────────────────────
OTHER_PEERS = [
    random_peer_id(),
    random_peer_id(),
    random_peer_id(),
    random_peer_id(),
]

ALL_PEERS = [MY_PEER_ID] + OTHER_PEERS

# ── Parámetros de protocolo (igual que en src/discovery.ts) ──────────────────
ANNOUNCE_INTERVAL = 3.0   # segundos entre broadcasts
PEER_TIMEOUT      = 10.0  # segundos sin oír a un peer → muerto
SIMULATION_TIME   = 40    # segundos totales de simulación
PACKET_LOSS       = 0.15  # fracción de paquetes UDP perdidos

print(f"Red: {len(ALL_PEERS)} peers")
for p in ALL_PEERS:
    label = " ← TÚ" if p == MY_PEER_ID else ""
    print(f"  {p}{label}")

## 3. Simulación de broadcast + descubrimiento

Cada peer envía un anuncio UDP cada `ANNOUNCE_INTERVAL` segundos.
Todos los demás lo reciben (con probabilidad `1 - PACKET_LOSS`).
Si un peer no se oye por `PEER_TIMEOUT` segundos, se elimina de la tabla.

Grabamos todos los eventos `(t, tipo, origen, destino)` para la gráfica.

In [ ]:
import random as rng
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class PeerState:
    peer_id: str
    known: dict = field(default_factory=dict)   # peerId → last_seen
    next_announce: float = 0.0

    def update(self, from_id: str, t: float):
        self.known[from_id] = t

    def gc(self, t: float, timeout: float) -> list:
        dead = [p for p, ts in self.known.items() if t - ts > timeout]
        for p in dead:
            del self.known[p]
        return dead

# ── Estado inicial ────────────────────────────────────────────────────────────
# Escalonamos el primer anuncio de cada peer para evitar que todos
# anuncien en el mismo instante t=0 (jitter de arranque).
states = {
    pid: PeerState(pid, next_announce=rng.uniform(0, ANNOUNCE_INTERVAL))
    for pid in ALL_PEERS
}

# ── Registro de eventos ───────────────────────────────────────────────────────
events = []  # (t, event_type, src, dst)
# event_type: 'up' | 'down' | 'announce'

# ── Loop de simulación (tick cada 0.5 s de tiempo simulado) ──────────────────
dt = 0.5
t = 0.0
while t <= SIMULATION_TIME:
    for pid, state in states.items():
        # ¿Le toca anunciar?
        if t >= state.next_announce:
            state.next_announce = t + ANNOUNCE_INTERVAL
            # Broadcast: cada otro peer recibe el anuncio (con pérdida)
            for other_id, other_state in states.items():
                if other_id == pid:
                    continue
                if rng.random() < PACKET_LOSS:
                    continue  # paquete perdido
                already_known = pid in other_state.known
                other_state.update(pid, t)
                if not already_known:
                    events.append((round(t, 2), 'up', pid, other_id))

        # GC: eliminar peers que llevan demasiado tiempo sin anunciar
        dead = state.gc(t, PEER_TIMEOUT)
        for d in dead:
            events.append((round(t, 2), 'down', d, pid))

    t = round(t + dt, 3)

# ── Resumen ───────────────────────────────────────────────────────────────────
ups   = [(t, s, d) for t, ev, s, d in events if ev == 'up']
downs = [(t, s, d) for t, ev, s, d in events if ev == 'down']
print(f"Eventos peer-up:   {len(ups)}")
print(f"Eventos peer-down: {len(downs)}")
print(f"\nPrimeros 10 peer-up:")
for t, src, dst in ups[:10]:
    src_label = src[:8] + (" (tú)" if src == MY_PEER_ID else "")
    dst_label = dst[:8] + (" (tú)" if dst == MY_PEER_ID else "")
    print(f"  t={t:5.1f}s  {dst_label} descubre a {src_label}")

## 4. Grafo de conectividad en el tiempo

Cada frame muestra el estado del grafo en un instante `t`:

- **Nodo verde**: tu propio peerId.
- **Nodo azul**: otro peer.
- **Arista**: peer A conoce a peer B (undirected: si A conoce B y B conoce A).
- **Arista punteada roja**: peer-down en este tick.

Se genera un snapshot por segundo de simulación.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

# ── Reconstruir estado del grafo en cada instante t ──────────────────────────
# Reproducimos la simulación de nuevo pero guardando snapshots cada segundo.

SNAPSHOT_INTERVAL = 1.0   # s de simulación entre frames

states2 = {
    pid: PeerState(pid, next_announce=rng.uniform(0, ANNOUNCE_INTERVAL))
    for pid in ALL_PEERS
}
rng.seed(42)  # reproducible

snapshots = []  # lista de grafos NetworkX
next_snap = 0.0
t = 0.0

while t <= SIMULATION_TIME:
    for pid, state in states2.items():
        if t >= state.next_announce:
            state.next_announce = t + ANNOUNCE_INTERVAL
            for other_id, other_state in states2.items():
                if other_id == pid or rng.random() < PACKET_LOSS:
                    continue
                other_state.update(pid, t)
        state.gc(t, PEER_TIMEOUT)

    if t >= next_snap:
        # Construir grafo no dirigido: arista si AMBOS se conocen mutuamente
        G = nx.Graph()
        G.add_nodes_from(ALL_PEERS)
        for pid, state in states2.items():
            for known_id in state.known:
                if pid in states2[known_id].known:  # mutuo
                    G.add_edge(pid, known_id)
        snapshots.append((round(t, 1), G.copy()))
        next_snap = round(next_snap + SNAPSHOT_INTERVAL, 3)

    t = round(t + dt, 3)

print(f"{len(snapshots)} snapshots generados")

# ── Posición fija de nodos (spring layout calculado una vez) ──────────────────
G_full = nx.complete_graph(len(ALL_PEERS))
G_full = nx.relabel_nodes(G_full, {i: p for i, p in enumerate(ALL_PEERS)})
pos = nx.spring_layout(G_full, seed=7)

# ── Dibujar grid de snapshots ─────────────────────────────────────────────────
frames_to_show = [0, 3, 8, 15, 25, len(snapshots) - 1]
frames_to_show = [i for i in frames_to_show if i < len(snapshots)]

cols = 3
rows = (len(frames_to_show) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for ax, idx in zip(axes, frames_to_show):
    snap_t, G = snapshots[idx]

    node_colors = ['#2ecc71' if n == MY_PEER_ID else '#3498db' for n in G.nodes()]
    node_sizes  = [600 if n == MY_PEER_ID else 400 for n in G.nodes()]
    labels      = {n: n[:6] + ('…' if len(n) > 6 else '') for n in G.nodes()}

    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=node_sizes)
    nx.draw_networkx_labels(G, pos, ax=ax, labels=labels, font_size=7, font_color='white')
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#95a5a6', width=1.5, alpha=0.8)

    connected = G.degree(MY_PEER_ID)
    ax.set_title(f"t = {snap_t}s  |  edges={G.number_of_edges()}  |  tu grado={connected}",
                 fontsize=9)
    ax.axis('off')

for ax in axes[len(frames_to_show):]:
    ax.axis('off')

green_patch = mpatches.Patch(color='#2ecc71', label='Tú')
blue_patch  = mpatches.Patch(color='#3498db', label='Otros peers')
fig.legend(handles=[green_patch, blue_patch], loc='lower right', fontsize=9)
fig.suptitle('Evolución del grafo de descubrimiento UDP', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 5. Gráfica de grado a lo largo del tiempo

El **grado** de un nodo = cuántos peers conoce (mutuamente) en cada instante.
Para tu peerId, muestra cuántos vecinos tienes a lo largo de la simulación.

In [ ]:
times       = [snap_t for snap_t, _ in snapshots]
my_degrees  = [G.degree(MY_PEER_ID) for _, G in snapshots]
avg_degrees = [sum(dict(G.degree()).values()) / max(G.number_of_nodes(), 1) for _, G in snapshots]
edge_counts = [G.number_of_edges() for _, G in snapshots]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(times, my_degrees, color='#2ecc71', linewidth=2, label=f'Tu grado ({MY_PEER_ID[:8]}…)')
ax1.plot(times, avg_degrees, color='#3498db', linewidth=1.5, linestyle='--', label='Grado promedio red')
ax1.set_ylabel('Grado (peers conocidos)')
ax1.set_ylim(-0.2, len(ALL_PEERS))
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)
ax1.set_title('Grado por peer a lo largo del tiempo')

ax2.fill_between(times, edge_counts, alpha=0.3, color='#9b59b6')
ax2.plot(times, edge_counts, color='#9b59b6', linewidth=2, label='Aristas totales')
ax2.axhline(len(ALL_PEERS) * (len(ALL_PEERS) - 1) // 2, color='gray',
            linestyle=':', label='Máximo posible (grafo completo)')
ax2.set_xlabel('Tiempo simulado (s)')
ax2.set_ylabel('Aristas')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ── Estadísticas finales ──────────────────────────────────────────────────────
_, G_final = snapshots[-1]
print(f"\nEstado final (t={SIMULATION_TIME}s):")
print(f"  Aristas:         {G_final.number_of_edges()} / {len(ALL_PEERS)*(len(ALL_PEERS)-1)//2} máximo")
print(f"  Tu grado:        {G_final.degree(MY_PEER_ID)}")
print(f"  Red conectada:   {nx.is_connected(G_final)}")
if G_final.number_of_edges() > 0:
    print(f"  Densidad:        {nx.density(G_final):.2f}")
    print(f"  Diámetro:        {nx.diameter(G_final) if nx.is_connected(G_final) else 'inf (desconectada)'}")

## 6. Preguntas para explorar

Cambia los parámetros de la celda 2 y vuelve a correr:

1. **¿Qué pasa con `PACKET_LOSS = 0.5`?** ¿La red llega a ser completa?
2. **¿Qué pasa con `ANNOUNCE_INTERVAL = 10` y `PEER_TIMEOUT = 11`?**
   Casi sin margen — los peers aparecen y desaparecen constantemente.
3. **Añade un peer que se une tarde** (`next_announce` grande) y observa
   cuánto tarda en que todos lo conozcan.
4. **¿Qué relación hay entre `PEER_TIMEOUT / ANNOUNCE_INTERVAL`
   y la resiliencia a pérdidas?** En `src/discovery.ts` la ratio es `10/3 ≈ 3.3`:
   el peer necesita fallar **3 broadcasts consecutivos** para ser expulsado.

El código real está en [src/discovery.ts](../../src/discovery.ts).
El notebook 01 profundiza en cada decisión de diseño.